### Transfer learning con modelos preentrenados

En el notebook anterior, nuestro modelo base aprendió a detectar bulos **desde cero** con solo 776 noticias. Ese modelo no sabe que *"Gobierno"* y *"Ejecutivo"* significan casi lo mismo, ni ha oído hablar nunca de la COVID-19.

El **transfer learning** (aprendizaje por transferencia) parte de otra idea: aprovechar un modelo que ya ha sido **preentrenado con millones de textos** y que, por tanto, ya "sabe español". Es como enseñar a detectar bulos a alguien que ya sabe leer, en lugar de enseñarle el idioma desde el principio con 776 ejemplos.

Hay dos maneras principales de hacerlo, y las probaremos las dos:

| Estrategia | ¿Qué hacemos con el modelo preentrenado? | Coste |
|---|---|---|
| **Extracción de características** | Lo usamos **congelado** para convertir cada texto en un vector (*embedding*) y entrenamos encima un clasificador clásico. | Bajo: funciona sin GPU |
| **Ajuste fino** (*fine-tuning*) | Añadimos una capa de clasificación y **reentrenamos todo el modelo** con nuestros datos. | Alto: conviene usar GPU |

Este notebook está estructurado como sigue:

- **BLOQUE A**: carga de datos y modelo base.
- **BLOQUE B**: transfer learning I, extracción de características con *embeddings*.
- **BLOQUE C**: transfer learning II, ajuste fino de BETO (un BERT en español).
- **BLOQUE D**: comparación de los tres modelos sobre las noticias nuevas.
- **BLOQUE E**: guardar el modelo y usarlo con noticias inventadas.

> **Antes de empezar en Google Colab**: ve a *Entorno de ejecución*, después *Cambiar tipo de entorno de ejecución*, y selecciona **GPU T4**. El bloque C es mucho más rápido con GPU.

In [ ]:
# Preparación del entorno: funciona tanto en Google Colab como en local
import os, sys

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB and not os.path.exists("seminario-manadine-transfer-learning"):
    !git clone -q https://github.com/Vins16/seminario-manadine-transfer-learning.git
    %pip install -q -r seminario-manadine-transfer-learning/requirements.txt

if EN_COLAB:
    RAIZ = "seminario-manadine-transfer-learning"
else:
    RAIZ = "." if os.path.exists("data") else ".."
sys.path.insert(0, RAIZ)
DATA = f"{RAIZ}/data"
MODELS = f"{RAIZ}/models"

In [ ]:
import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.pipeline import Pipeline

from utils import preprocesar

sns.set_theme(style="whitegrid")

In [ ]:
# ¿Tenemos GPU disponible?
HAY_GPU = torch.cuda.is_available()
dispositivo = "cuda" if HAY_GPU else "cpu"
print(f"Usando: {dispositivo}")

### BLOQUE A: carga de datos y modelo base

Usamos **exactamente la misma partición** que en el notebook 3 para que la comparación sea justa.

In [ ]:
df = pd.read_csv(f"{DATA}/noticias_2018_particion.csv")
nuevas = pd.read_csv(f"{DATA}/noticias_2020_21.csv")

train = df[df["particion"] == "train"].reset_index(drop=True)
val = df[df["particion"] == "val"].reset_index(drop=True)

print(f"Entrenamiento: {len(train)} | Validación: {len(val)} | Nuevas: {len(nuevas)}")

Volvemos a entrenar el modelo base del notebook 3 (TF-IDF + regresión logística). Tarda un segundo y así no dependemos de que las versiones de las librerías coincidan con las del fichero guardado.

In [ ]:
baseline = Pipeline([
    ("tfidf", TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=2)),
    ("modelo", LogisticRegression(max_iter=1000, C=10)),
])
baseline.fit(train["texto"].apply(preprocesar), train["target"])

pred_val_base = baseline.predict(val["texto"].apply(preprocesar))
print(f"Modelo base - accuracy en validación: {accuracy_score(val['target'], pred_val_base):.2f}")

### BLOQUE B: transfer learning I, extracción de características con *embeddings*

Un ***embedding*** es un vector de números que representa el **significado** de un texto. Textos con significados parecidos tienen vectores cercanos, aunque no compartan ninguna palabra. Esa es la gran diferencia con TF-IDF, que solo ve qué palabras exactas aparecen.

Usaremos [`paraphrase-multilingual-MiniLM-L12-v2`](https://huggingface.co/sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2), un modelo de la librería **sentence-transformers** preentrenado en más de 50 idiomas, que convierte cada texto en un vector de 384 dimensiones.

In [ ]:
from sentence_transformers import SentenceTransformer

modelo_emb = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2", device=dispositivo)

Antes de clasificar noticias, juguemos un poco con el modelo. ¿Qué frases se parecen más entre sí?

In [ ]:
frases = [
    "El Gobierno aprueba una nueva ley de vivienda",
    "El Congreso da luz verde a la normativa sobre alquileres",
    "El Real Madrid gana la Champions League",
]

emb_frases = modelo_emb.encode(frases)
print("Dimensiones:", emb_frases.shape)

# Similitud del coseno entre cada par de frases (1 = significado idéntico)
similitud = modelo_emb.similarity(emb_frases, emb_frases)
pd.DataFrame(similitud.numpy(), index=frases, columns=["Frase 1", "Frase 2", "Frase 3"]).round(2)

Las frases 1 y 2 no comparten casi ninguna palabra, pero el modelo sabe que hablan de lo mismo. Para TF-IDF serían prácticamente independientes.

Ahora convertimos todas nuestras noticias en *embeddings*. El modelo procesa como máximo los primeros 128 *tokens* de cada texto (unas 100 palabras), es decir, el titular y el comienzo de la noticia.

In [ ]:
X_train_emb = modelo_emb.encode(train["texto"].tolist(), batch_size=32, show_progress_bar=True)
X_val_emb = modelo_emb.encode(val["texto"].tolist(), batch_size=32, show_progress_bar=True)
X_nuevas_emb = modelo_emb.encode(nuevas["texto"].tolist(), batch_size=32, show_progress_bar=True)

X_train_emb.shape

Con los *embeddings* como variables explicativas, entrenamos una regresión logística: el mismo clasificador que en el modelo base, pero con una representación del texto mucho más rica.

In [ ]:
clf_emb = LogisticRegression(max_iter=1000)
clf_emb.fit(X_train_emb, train["target"])

pred_val_emb = clf_emb.predict(X_val_emb)
print(f"Accuracy: {accuracy_score(val['target'], pred_val_emb):.2f}\n")
print(classification_report(val["target"], pred_val_emb, target_names=["verdadera", "falsa"], zero_division=0))

### BLOQUE C: transfer learning II, ajuste fino (*fine-tuning*) de BETO

**BERT** (*Bidirectional Encoder Representations from Transformers*) es un modelo de lenguaje basado en la arquitectura *Transformer*, desarrollado por Google. Se preentrena con una tarea muy sencilla: adivinar palabras ocultas en millones de frases. Al hacerlo, aprende gramática, vocabulario y bastante conocimiento del mundo.

**[BETO](https://github.com/dccuchile/beto)** es un BERT entrenado por la Universidad de Chile **exclusivamente con textos en español**. Para adaptarlo a nuestra tarea:

1. Le añadimos encima una pequeña **capa de clasificación** (falsa / verdadera), que empieza con pesos aleatorios.
2. Entrenamos **todo el modelo** durante pocas épocas y con una tasa de aprendizaje muy baja, para ajustar sus conocimientos sin "olvidar" lo que ya sabe.

In [ ]:
from datasets import Dataset
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments)

NOMBRE_MODELO = "dccuchile/bert-base-spanish-wwm-cased"

#### El *tokenizer*

Los modelos tipo BERT no trabajan con palabras completas, sino con ***tokens***: fragmentos de palabra. Las palabras frecuentes son un único token y las raras se dividen en trozos (los que empiezan por `##` continúan el token anterior). Gracias a esto, el modelo puede representar cualquier palabra, aunque no la haya visto nunca... como *"COVID"*.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(NOMBRE_MODELO)

for frase in ["La desinformación se propaga rápidamente", "Vacunas contra la COVID-19"]:
    print(tokenizer.tokenize(frase))

In [ ]:
# El tokenizer convierte cada token en un número (su posición en el vocabulario del modelo)
tokenizer("Hola MANADINE")

Convertimos nuestros DataFrames al formato `Dataset` de Hugging Face y tokenizamos todos los textos. El modelo admite hasta 512 tokens, pero usaremos 256 para que el entrenamiento sea más rápido (128 si no hay GPU).

In [ ]:
MAX_TOKENS = 256 if HAY_GPU else 128


def a_dataset(datos):
    # Hugging Face espera que la columna con la etiqueta se llame "labels"
    return Dataset.from_pandas(datos[["texto", "target"]].rename(columns={"target": "labels"}), preserve_index=False)


def tokenizar(lote):
    return tokenizer(lote["texto"], truncation=True, max_length=MAX_TOKENS)


ds_train = a_dataset(train).map(tokenizar, batched=True)
ds_val = a_dataset(val).map(tokenizar, batched=True)
ds_nuevas = a_dataset(nuevas).map(tokenizar, batched=True)

if not HAY_GPU:
    print("Sin GPU: entrenaremos solo con 200 noticias para que no tarde demasiado.")
    ds_train = ds_train.shuffle(seed=42).select(range(200))

ds_train

#### El modelo

Cargamos BETO con una capa de clasificación de **2 clases**. Verás un aviso diciendo que algunos pesos se han inicializado de forma aleatoria: es lo esperado, se trata precisamente de la nueva capa de clasificación que vamos a entrenar.

In [ ]:
modelo_beto = AutoModelForSequenceClassification.from_pretrained(
    NOMBRE_MODELO,
    num_labels=2,
    id2label={0: "verdadera", 1: "falsa"},
    label2id={"verdadera": 0, "falsa": 1},
)
print(f"Parámetros del modelo: {modelo_beto.num_parameters() / 1e6:.0f} millones")

In [ ]:
def calcular_metricas(eval_pred):
    logits, etiquetas = eval_pred
    predicciones = np.argmax(logits, axis=-1)   # la clase con mayor puntuación
    return {"accuracy": accuracy_score(etiquetas, predicciones), "f1": f1_score(etiquetas, predicciones)}

#### Entrenamiento

La clase `Trainer` de Hugging Face se encarga de todo el bucle de entrenamiento. Los hiperparámetros más importantes son:

- **`num_train_epochs`**: número de veces que el modelo recorre todos los datos de entrenamiento. En *fine-tuning* bastan unas pocas.
- **`learning_rate`**: tamaño de los pasos con los que se actualizan los pesos. Debe ser **muy pequeño** (del orden de 2e-5) para no destruir lo que el modelo aprendió en el preentrenamiento.
- **`per_device_train_batch_size`**: número de noticias que se procesan a la vez.

In [ ]:
argumentos = TrainingArguments(
    output_dir=f"{MODELS}/checkpoints",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",       # evaluamos en validación al final de cada época
    save_strategy="no",
    logging_steps=10,
    fp16=HAY_GPU,                # cálculos en media precisión: más rápido en GPU
    report_to="none",
    seed=42,
)

trainer = Trainer(
    model=modelo_beto,
    args=argumentos,
    train_dataset=ds_train,
    eval_dataset=ds_val,
    processing_class=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=calcular_metricas,
)

In [ ]:
# ¡A entrenar! Con GPU T4 tarda unos pocos minutos
trainer.train()

In [ ]:
# Resultados sobre el conjunto de validación
salida_val = trainer.predict(ds_val)
pred_val_beto = np.argmax(salida_val.predictions, axis=-1)

print(f"Accuracy: {accuracy_score(val['target'], pred_val_beto):.2f}\n")
print(classification_report(val["target"], pred_val_beto, target_names=["verdadera", "falsa"], zero_division=0))

### BLOQUE D: comparación de los tres modelos sobre las noticias nuevas

Llega el momento de la verdad: ¿qué modelo generaliza mejor a noticias publicadas años después y sobre temas que no ha visto?

In [ ]:
pred_nuevas_base = baseline.predict(nuevas["texto"].apply(preprocesar))
pred_nuevas_emb = clf_emb.predict(X_nuevas_emb)
pred_nuevas_beto = np.argmax(trainer.predict(ds_nuevas).predictions, axis=-1)

In [ ]:
predicciones = {
    "TF-IDF + Reg. logística": (pred_val_base, pred_nuevas_base),
    "Embeddings + Reg. logística": (pred_val_emb, pred_nuevas_emb),
    "BETO (fine-tuning)": (pred_val_beto, pred_nuevas_beto),
}

filas = []
for modelo, (p_val, p_nuevas) in predicciones.items():
    for conjunto, y_real, p in [("Validación 2018", val["target"], p_val), ("Nuevas 2020-21", nuevas["target"], p_nuevas)]:
        filas.append({"modelo": modelo, "conjunto": conjunto,
                      "accuracy": accuracy_score(y_real, p), "f1": f1_score(y_real, p)})

resultados = pd.DataFrame(filas)
resultados.pivot(index="modelo", columns="conjunto", values=["accuracy", "f1"]).round(2)

In [ ]:
plt.figure(figsize=(9, 5))
ax = sns.barplot(data=resultados, x="modelo", y="accuracy", hue="conjunto")
for contenedor in ax.containers:
    ax.bar_label(contenedor, fmt="%.2f")
plt.ylim(0, 1)
plt.axhline(0.5, ls="--", c="grey", label="Azar")
plt.title("Accuracy por modelo y conjunto")
plt.xlabel("")
plt.legend(loc="lower right")
plt.show()

**Para discutir:**

- ¿Qué modelo pierde menos rendimiento al pasar de 2018 a 2020-21? ¿Por qué crees que ocurre?
- ¿Compensa el coste de cómputo del *fine-tuning* frente a la extracción de *embeddings*?
- ¿En qué temas se equivoca más cada modelo? (pista: usa `nuevas["Topic"]`)

In [ ]:
# Tasa de acierto de BETO en las noticias nuevas, por tema
nuevas["acierto_beto"] = (pred_nuevas_beto == nuevas["target"])
nuevas.groupby("Topic")["acierto_beto"].agg(["mean", "count"]).sort_values("count", ascending=False).round(2)

In [ ]:
# Algunos ejemplos donde BETO se equivoca
errores = nuevas[~nuevas["acierto_beto"]]
for _, fila in errores.head(3).iterrows():
    real = "FALSA" if fila["target"] == 1 else "VERDADERA"
    print(f"[{fila['Topic']}] Realmente {real}:\n{fila['texto'][:300]}...\n")

### BLOQUE E: guardar el modelo y usarlo con noticias inventadas

Guardamos el modelo ajustado y lo cargamos con un `pipeline`, la forma más sencilla de usar un modelo de Hugging Face para hacer predicciones.

In [ ]:
RUTA_MODELO = f"{MODELS}/beto-bulos"
trainer.save_model(RUTA_MODELO)

In [ ]:
from transformers import pipeline

detector = pipeline("text-classification", model=RUTA_MODELO, device=dispositivo)

In [ ]:
noticias_inventadas = [
    "ÚLTIMA HORA: Científicos confirman que beber agua con limón cura todas las enfermedades. Los médicos no quieren que lo sepas. ¡¡COMPARTE ANTES DE QUE LO BORREN!!",
    "El Banco Central Europeo ha decidido mantener los tipos de interés sin cambios en su reunión de este jueves, según ha informado la institución en un comunicado.",
]

for noticia, resultado in zip(noticias_inventadas, detector(noticias_inventadas, truncation=True)):
    print(f"{resultado['label'].upper():<10} ({resultado['score']:.0%})  {noticia[:80]}...")

### Reto final

Escribe tu propia noticia (verdadera o inventada) y comprueba qué opina el modelo. ¿Eres capaz de engañarlo? ¿Qué cambios en el texto le hacen cambiar de opinión?

In [ ]:
mi_noticia = "Escribe aquí tu noticia"
detector(mi_noticia, truncation=True)

### Conclusiones

- Un modelo clásico (TF-IDF) depende del **vocabulario exacto** que vio al entrenar, y sufre mucho cuando cambian los temas.
- El **transfer learning** reutiliza el conocimiento de modelos preentrenados con enormes cantidades de texto. Con muy pocos datos propios se consiguen representaciones más ricas y, en general, más robustas.
- La **extracción de *embeddings*** es barata y funciona sin GPU. El ***fine-tuning*** suele dar mejores resultados, pero requiere más cómputo.